# 프롬프트·Chain·대화 이력 · v1.03

**2일차 2교시 · 수업 50분 + 휴식/전환 10분**

**목표:** 프롬프트 변수를 채우고, 이전 메시지를 다시 전달하는 챗봇을 만듭니다.

[전체 안내](README.md) · [Python 코드 읽기](../04.LLM_v2/PYTHON_GUIDE.md)

슬라이드로 개념을 먼저 보고 이 노트북을 실행합니다. **필수** 절을 위에서 아래로 실행하세요. 커널은 **Python (Day2 LangChain)**입니다. 다른 노트북의 변수는 자동으로 전달되지 않습니다.

셀 왼쪽 `[*]`는 실행 중이라는 뜻입니다. 응답을 기다린 뒤 다음 셀을 실행하세요. 요청 셀을 재실행하면 서버에 다시 요청합니다. 오류가 나면 그 셀에서 멈추고 오류 원문과 입력을 확인합니다.

**대응 슬라이드:** 19~24쪽 · [PDF 열기](../슬라이드/강의자료_2일차_v2.pdf#page=19)


## 준비 · 경로와 모델 객체

`Path`는 파일 경로를 다루는 도구입니다. 아래 셀은 현재 폴더부터 상위 폴더를 살펴 교재 위치를 찾습니다. `sys.path`에는 교재의 Python 모듈 위치를 추가합니다. 준비 코드는 제공하며 직접 외우지 않아도 됩니다.


In [ ]:
from pathlib import Path
import json
import os
import sys
from IPython.display import display, Image

# 현재 폴더에 교재가 없으면 부모 폴더를 차례로 확인합니다.
ROOT = Path.cwd().resolve()
while not (ROOT / "04.LLM_v2").is_dir():
    if ROOT == ROOT.parent:
        raise RuntimeError("NEPES 교재 폴더 안에서 노트북을 여세요.")
    ROOT = ROOT.parent
# import가 교재의 공통 모듈을 찾을 수 있도록 경로를 등록합니다.
sys.path.insert(0, str(ROOT / "04.LLM_v2"))
sys.path.insert(0, str(ROOT / "05.VLM_v2"))
print("교재 위치:", ROOT)
print("현재 Python:", sys.executable)


### 사용 전 API 안내 · 모델과 메시지

객체 만들기와 실제 실행을 구분해서 읽으세요. 아래 이름은 코드에 그대로 등장합니다.

| 함수·클래스 | 초기화·호출 예 | 입력 → 반환 / 주의점 |
|---|---|---|
| `ChatOllama` | `model = ChatOllama(model=이름, base_url=주소, temperature=0)` | Ollama 연결 객체. 생성만으로 추론하지 않습니다. `num_ctx`는 문맥 한도, `num_predict`는 출력 한도입니다. |
| `SystemMessage`, `HumanMessage`, `AIMessage` | `HumanMessage(content="질문")` | 역할별 메시지 객체. 시스템 지침 / 사용자 입력 / 모델 응답을 구분합니다. |
| `model.invoke` | `reply = model.invoke([HumanMessage("질문")])` | 메시지 목록 → `AIMessage`. 이때 서버에 요청합니다. `reply.content`는 본문, `response_metadata`는 실행 정보입니다. |

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# model에는 가중치가 아니라 서버 요청을 위한 Python 객체가 들어갑니다.
# 생성자만 실행해도 대화가 시작되는 것은 아닙니다. invoke가 요청합니다.
model = ChatOllama(
    model="gemma4:e2b-it-qat",
    base_url="http://127.0.0.1:11434",  # 수업은 host network를 사용합니다.
    temperature=0,                     # 비교 조건을 고정합니다. 정답 보장은 아닙니다.
    reasoning=False,                   # 수업에서는 최종 답변을 관찰합니다.
    num_ctx=4096,                      # 문맥 창 크기 설정: 글자 수와 다릅니다.
    num_predict=512,                   # 출력 토큰 상한: 잘리면 상한과 입력을 확인합니다.
    client_kwargs={"timeout": 180},    # 서버 응답을 기다리는 최대 시간(초).
)
print("사용할 모델:", model.model)


## 1. Python 사전으로 프롬프트 변수를 채웁니다

`{"question": "센서란?"}`는 키와 값을 묶은 사전입니다. `question` 키의 값이 템플릿의 `{question}` 자리에 들어갑니다.

`prompt.invoke`는 메시지를 만드는 단계입니다. 여기서는 모델을 호출하지 않습니다. 같은 invoke라는 이름이어도 호출하는 객체의 역할이 다릅니다.

![1. Python 사전으로 프롬프트 변수를 채웁니다](assets/lesson-2-6.svg)


### 사용 전 API 안내 · 프롬프트와 Chain

객체 만들기와 실제 실행을 구분해서 읽으세요. 아래 이름은 코드에 그대로 등장합니다.

| 함수·클래스 | 초기화·호출 예 | 입력 → 반환 / 주의점 |
|---|---|---|
| `ChatPromptTemplate.from_messages` | `prompt = ChatPromptTemplate.from_messages([("human", "{question}")])` | 역할·템플릿 목록 → 프롬프트 객체. 중괄호는 입력 변수 자리입니다. |
| `prompt.invoke` / `to_messages` | `prompt.invoke({"question": "질문"}).to_messages()` | 변수 사전 → 메시지 목록. 여기까지는 모델 추론이 없습니다. |
| `MessagesPlaceholder` | `MessagesPlaceholder("history")` | 프롬프트 안에 이전 메시지 목록이 들어갈 자리. 자동 저장 기능은 없습니다. |
| Chain 연결 / 실행 | `chain = prompt ⎮ model`; `chain.invoke(입력사전)` | 코드에서는 세로 막대 `&#124;`로 연결합니다. 실행 결과는 모델의 `AIMessage`입니다. |

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "한국어로 간결하게 답하세요."),
    ("human", "{question}"),  # 중괄호 안의 이름은 나중에 채울 변수입니다.
])
inputs = {"question": "온도 센서의 역할은 무엇인가요?"}
value = prompt.invoke(inputs)
for message in value.to_messages():  # 실제 전송할 메시지 목록으로 바꿉니다.
    print(message.type, message.content)


## 2. 파이프 연산자로 처리 순서를 연결합니다

`|`는 왼쪽 반환값을 오른쪽 입력으로 연결합니다. `chain = prompt | model`은 연결만 구성합니다. `chain.invoke(inputs)`를 실행하면 프롬프트 생성 다음에 모델 요청이 일어납니다.

입력 사전의 키가 틀리면 모델 요청 전에 오류가 납니다. `question`을 `questions`로 바꾸면 어떤 단계가 실패할지 먼저 예상해보세요.

![2. 파이프 연산자로 처리 순서를 연결합니다](assets/lesson-2-7.svg)


In [ ]:
chain = prompt | model
reply = chain.invoke(inputs)  # 실제 모델 호출 1회.
print(reply.content)


## 3. 기억은 이전 메시지를 다시 보내는 것입니다

`MessagesPlaceholder("history")`는 메시지 목록이 들어갈 자리입니다. 자동 저장소가 아닙니다. 매 요청에서 Python이 history를 전달해야 합니다.

첫 질문에는 작업 코드가 있지만 두 번째 질문만 보내면 그 코드를 알 수 없습니다. 아래는 같은 두 턴을 이력 없이 실행합니다.

![3. 기억은 이전 메시지를 다시 보내는 것입니다](assets/lesson-2-8.svg)


In [ ]:
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "한국어로 짧게 답하세요. 입력에 없는 작업 코드는 추측하지 마세요."),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])
chat_chain = chat_prompt | model
questions = ["작업 코드는 BLUE-27입니다. 확인해주세요.", "앞서 알려준 작업 코드는 무엇인가요?"]
for question in questions:
    reply = chat_chain.invoke({"history": [], "question": question})
    print(question, "→", reply.content)


## 4. TODO · 최근 대화만 선택합니다

아래 함수의 `return []`를 수정하세요. `enabled=False`이면 빈 목록, True이면 최근 `max_pairs`개의 질문·답 쌍을 반환합니다. 한 쌍은 메시지 두 개입니다.

`def`는 함수를 정의합니다. 정의만으로 함수 안의 코드가 실행되지는 않습니다. `return`은 호출한 곳으로 값을 돌려줍니다. `history[-4:]`는 뒤에서 최대 네 항목을 선택합니다. 아래에서 완성한 함수를 실제 챗봇에 사용합니다.


In [ ]:
def select_recent(history, max_pairs=2, enabled=True):
    # 이 실습의 history는 질문과 답이 번갈아 들어 있는 완전한 쌍 목록입니다.
    if type(max_pairs) is not int or max_pairs < 1:
        raise ValueError("max_pairs는 1 이상의 정수여야 합니다.")
    if not enabled:
        return []
    # TODO: 최근 max_pairs * 2개의 메시지를 선택하도록 바꾸세요.
    return []


In [ ]:
# 참고 정답: 먼저 위 TODO를 시도한 뒤 비교하세요.
# 다른 함수 이름을 사용하므로 실행해도 학생의 select_recent을 덮어쓰지 않습니다.
def select_recent_solution(history, max_pairs=2, enabled=True):
    if type(max_pairs) is not int or max_pairs < 1:
        raise ValueError("max_pairs는 1 이상의 정수여야 합니다.")
    if not enabled:
        return []  # 기억 기능을 끄면 과거 메시지를 보내지 않습니다.
    message_count = max_pairs * 2  # 질문 한 개와 답 한 개가 한 쌍입니다.
    return history[-message_count:]  # 음수 시작 위치: 목록 끝에서부터 선택합니다.

assert select_recent_solution(["질문1", "답1", "질문2", "답2"], 1) == ["질문2", "답2"]

정답을 확인한 뒤 **위 학생 함수에 반영하고 다시 실행**하세요. 아래 검사는 학생 함수를 검사합니다. 참고 정답 셀만 실행해서는 TODO가 완료되지 않습니다.

**확인:** 아래 첫 검사는 TODO가 미완성이면 실패하는 것이 정상입니다. 직접 수정 후 실행하세요. 강사 검증 환경에서만 `DAY2_TODO_VARIANT=solution`으로 참고 구현을 선택합니다.


In [ ]:
# 참고 구현 선택은 자동 검증 전용입니다. 기본 수업에서는 위 학생 함수를 사용합니다.
if os.environ.get("DAY2_TODO_VARIANT") == "solution":
    select_recent = select_recent_solution

sample = [HumanMessage("첫 질문"), AIMessage("첫 답"), HumanMessage("둘째 질문"), AIMessage("둘째 답")]
assert select_recent(sample, 1) == sample[-2:]
assert select_recent(sample, enabled=False) == []
print("최근 대화 선택 검사 통과")


In [ ]:
history = []  # 셀을 다시 실행하면 이 챗봇의 대화가 초기화됩니다.
for question in questions:
    recent = select_recent(history, max_pairs=2)
    inputs = {"history": recent, "question": question}
    # 모델에 실제로 보내는 이전 대화를 먼저 확인합니다.
    print("전달 이력:", [(item.type, item.content) for item in recent])
    reply = chat_chain.invoke(inputs)
    if not reply.content or reply.response_metadata.get("done_reason") == "length":
        raise RuntimeError("빈 응답 또는 잘린 응답입니다. 이력에 추가하지 않습니다.")
    # 성공한 질문·답을 함께 추가합니다. invoke에서 예외가 나면 이 줄에 도달하지 않습니다.
    history.extend([HumanMessage(content=question), reply])
    print(reply.content)


## 5. 기억·지시·모델을 구분해 평가합니다

대화 이력은 입력 자료이고, “조건을 확인하고 짧은 근거를 쓰라”는 문장은 출력 지시입니다. 둘 다 모델의 정답을 보장하지 않습니다.

**결과 비교 5분:** 위에서 실행한 이력 없는 응답과 이력 있는 응답을 나란히 읽습니다. 실제 입력에 작업 코드가 포함되었는지, 답변이 그 코드와 일치하는지 확인하세요. 한 번의 응답으로 모델의 일반 성능을 판단하지 않습니다.


In [ ]:
# clear는 Python 목록만 비웁니다. 모델 파일이나 저장 로그를 지우지 않습니다.
history.clear()
print("초기화한 대화:", history)


## 완료 확인 · 직접 설명해보세요

- [ ] prompt.invoke와 model.invoke를 구분했다.
- [ ] TODO 함수가 실제 챗봇에 적용되는 위치를 찾았다.
- [ ] 이력 없는 요청과 있는 요청의 실제 메시지를 비교했다.

실행 성공과 의미 정답을 구분하세요. 답변이 이상하면 원문을 남기고, 어떤 입력 또는 단계에서 문제가 생겼는지 표시합니다.
